# Regular Expressions

Chapter 2 wrote two small parsers with plain string methods. An Instagram caption was searched for a mention:

```python
mention_start = caption.find("@")
mention = caption[mention_start:]
```

and a Google search box was turned into a URL slug:

```python
url_slug = search_query.lower().replace(" ", "-")
```

Both work on the one example each was written for. This chapter starts by feeding them a second example, and then builds the tool that handles both.

A **regular expression**, or regex, is a small pattern language for describing text. You write what the text looks like, and Python finds every place that matches. It is built into the standard library as the `re` module, so there is nothing to install, and it turns up everywhere in data work: cleaning columns, pulling fields out of log lines, validating input, and splitting text that a simple `.split()` cannot handle.

**What we will learn:**

1. Where chapter 2's string methods run out
2. Raw strings, and why every pattern begins with `r`
3. The five `re` functions you will actually use
4. Character classes, quantifiers and anchors
5. Groups, and named groups that keep patterns readable
6. Greedy and lazy matching
7. `re.compile` and flags
8. When a regex is the wrong tool

---
# 1. Where Chapter 2's Code Runs Out

Here is chapter 2's caption parser, given a caption with more than one hashtag and more than one mention.

In [1]:
caption = "Learning #Python and #pandas! Thanks @guido and @raymondh for the help. #100DaysOfCode"

has_python = "#Python" in caption          # chapter 2's hashtag check
mention_start = caption.find("@")          # chapter 2's mention finder
mention = caption[mention_start:]

print("Hashtag present:", has_python)
print("User mentioned :", mention)

Hashtag present: True
User mentioned : @guido and @raymondh for the help. #100DaysOfCode


Two problems. The mention is not `@guido`, it is everything from the first `@` to the end of the caption. And `"#Python" in caption` only answers a question about one hashtag you already knew the name of, which is not the question Instagram needs answered.

Finding every hashtag with string methods means writing the scan yourself:

In [2]:
def find_hashtags(text):
    tags = []
    word = ""

    for char in text + " ":                # the trailing space closes the last word
        if char == "#":
            word = "#"
        elif word and (char.isalnum() or char == "_"):
            word += char
        elif word:
            tags.append(word)
            word = ""

    return tags


print(find_hashtags(caption))

['#Python', '#pandas', '#100DaysOfCode']


Ten lines, three branches, and a trailing space bolted on so the last hashtag is not dropped. It also has a bug:

In [3]:
print("with a space   :", find_hashtags("#python #pandas"))
print("without a space:", find_hashtags("#python#pandas"))

with a space   : ['#python', '#pandas']
without a space: ['#pandas']


One hashtag short, and nothing said so. That is the real cost of writing the scan by hand: not the length, but the cases you did not think of.

Here is the same job:

In [4]:
import re

print(re.findall(r"#\w+", caption))
print(re.findall(r"@\w+", caption))
print(re.findall(r"#\w+", "#python#pandas"))    # the case the loop got wrong

['#Python', '#pandas', '#100DaysOfCode']
['@guido', '@raymondh']
['#python', '#pandas']


One line each. `#\w+` reads as "a `#`, then one or more word characters", and `findall` returns every stretch of text that fits.

The URL slug has the same shape of problem. Chapter 2's version assumed the query was clean:

In [5]:
query = "How to learn Python in 2026?  (beginner guide)"

print(query.lower().replace(" ", "-"))

how-to-learn-python-in-2026?--(beginner-guide)


A question mark and a pair of brackets survived into the URL, and the double space became a double hyphen. Fixing that with `.replace()` means one call per punctuation mark you can think of, and you will not think of them all.

The regex version says the rule instead of listing the exceptions: replace every run of characters that is not a letter or digit with a single hyphen.

In [6]:
slug = re.sub(r"[^a-z0-9]+", "-", query.lower()).strip("-")

print(slug)

how-to-learn-python-in-2026-beginner-guide


---
# 2. Raw Strings, and Why Every Pattern Starts With `r`

Every pattern above was written `r"..."`. That prefix makes a **raw string**, in which a backslash is just a backslash.

It matters because both Python and the regex engine use backslashes, and without the `r` Python gets there first.

In [7]:
print("normal string:", repr("\b"), "length", len("\b"))
print("raw string   :", repr(r"\b"), "length", len(r"\b"))

normal string: '\x08' length 1
raw string   : '\\b' length 2


In a normal string, `\b` is one character: a backspace, left over from the days of teletypes. In a raw string it is two characters, a backslash and a `b`, which is what the regex engine needs to see in order to read it as "word boundary".

This is not a style preference. Leave the `r` off and the pattern silently stops working:

In [8]:
text = "I like Python"

print("without r:", re.findall("\bPython", text))
print("with r   :", re.findall(r"\bPython", text))

without r: []
with r   : ['Python']


No error, no warning, just an empty list. Python turned `\b` into a backspace character before `re` ever saw the pattern, and the text contains no backspaces.

Write `r"..."` on every pattern, including the ones with no backslash in them yet. You will add one eventually.

---
# 3. The Five Functions

The `re` module has plenty in it. These five do almost all the work.

| Function | Returns | Use it for |
|---|---|---|
| `re.search` | the first match, or `None` | "is it in there, and where?" |
| `re.findall` | a list of strings | "give me all of them" |
| `re.finditer` | an iterator of match objects | all of them, with positions |
| `re.sub` | a new string | find and replace |
| `re.split` | a list of strings | splitting on a pattern |

### `re.search`: the first match, or nothing

In [9]:
match = re.search(r"@\w+", caption)

if match:
    print(match)
    print("matched text:", match.group())
    print("position    :", match.start(), "to", match.end())
    print("span        :", match.span())

<re.Match object; span=(37, 43), match='@guido'>
matched text: @guido
position    : 37 to 43
span        : (37, 43)


That `if` is not decoration. `re.search` returns `None` when nothing matches, a match object is truthy and `None` is falsy, so an `if` is the normal way to use it. Reaching for `.group()` without the check is the most common way to meet `AttributeError: 'NoneType' object has no attribute 'group'`:

In [10]:
for pattern in [r"@\w+", r"@[0-9]+"]:
    found = re.search(pattern, caption)
    if found:
        print(f"{pattern:<10} matched {found.group()!r}")
    else:
        print(f"{pattern:<10} no match")

@\w+       matched '@guido'
@[0-9]+    no match


`re.match` is a different function, and the name misleads everybody once. It only looks at the **start** of the string. `re.fullmatch` requires the whole string to match, which is what you want when validating a field:

In [11]:
print("search    'Python'  :", re.search(r"Python", caption))
print("match     'Python'  :", re.match(r"Python", caption))
print("match     'Learning':", re.match(r"Learning", caption))
print()
print("fullmatch '2026'    :", re.fullmatch(r"\d{4}", "2026"))
print("fullmatch '2026a'   :", re.fullmatch(r"\d{4}", "2026a"))

search    'Python'  : <re.Match object; span=(10, 16), match='Python'>
match     'Python'  : None
match     'Learning': <re.Match object; span=(0, 8), match='Learning'>

fullmatch '2026'    : <re.Match object; span=(0, 4), match='2026'>
fullmatch '2026a'   : None


### `re.findall` and `re.finditer`

`findall` gives you the matched text and nothing else. `finditer` gives you match objects, so you also get positions, and it is lazy in chapter 12's sense: matches are found as you ask for them.

In [12]:
for found in re.finditer(r"#\w+", caption):
    print(f"  {found.group():<16} at characters {found.start()}-{found.end()}")

  #Python          at characters 9-16
  #pandas          at characters 21-28
  #100DaysOfCode   at characters 72-86


### `re.sub`: find and replace

`sub` is `str.replace()` with a pattern instead of a fixed string. Masking data before it goes into a log or a shared notebook is the everyday use:

In [13]:
record = "Card 4111 1111 1111 1234, phone 98765 43210"

print(re.sub(r"\d", "X", record))
print(re.sub(r"\d{4,}", "[redacted]", record))

Card XXXX XXXX XXXX XXXX, phone XXXXX XXXXX
Card [redacted] [redacted] [redacted] [redacted], phone [redacted] [redacted]


The replacement can also be a **function**, which receives each match and returns its replacement. That covers everything a fixed string cannot:

In [14]:
prices = "Books $59.98, Electronics $45.00, Appliances $89.99"

def add_tax(found):
    amount = float(found.group(1))
    return f"${amount * 1.18:.2f}"

print(re.sub(r"\$(\d+\.\d\d)", add_tax, prices))

Books $70.78, Electronics $53.10, Appliances $106.19


### `re.split`: splitting on a pattern

`str.split()` takes one separator. `re.split` takes a pattern, which is what you need when the separator varies:

In [15]:
messy = "Books;Electronics, Appliances|Toys , Garden"

print("str.split:", messy.split(","))
print("re.split :", re.split(r"\s*[;,|]\s*", messy))

str.split: ['Books;Electronics', ' Appliances|Toys ', ' Garden']
re.split : ['Books', 'Electronics', 'Appliances', 'Toys', 'Garden']


---
# 4. The Building Blocks

A pattern is built from three kinds of piece: what to match, how many, and where.

### What to match

| Pattern | Matches |
|---|---|
| `\d` | a digit, `0-9` |
| `\w` | a word character: letter, digit or underscore |
| `\s` | whitespace: space, tab, newline |
| `\D` `\W` `\S` | the opposite of each |
| `.` | any character except a newline |
| `[abc]` | one of `a`, `b`, `c` |
| `[a-z]` | any lowercase letter |
| `[^abc]` | any character that is **not** `a`, `b` or `c` |

The capital letter is always the negation, and `^` inside square brackets means "not".

In [16]:
sample = "Order A001 shipped on 2024-03-01 for $59.98"

print("digits      :", re.findall(r"\d", sample)[:6], "...")
print("numbers     :", re.findall(r"\d+", sample))
print("words       :", re.findall(r"\w+", sample)[:4], "...")
print("not digits  :", re.findall(r"[^\d\s]+", sample))
print("vowels      :", re.findall(r"[aeiou]", sample))

digits      : ['0', '0', '1', '2', '0', '2'] ...
numbers     : ['001', '2024', '03', '01', '59', '98']
words       : ['Order', 'A001', 'shipped', 'on'] ...
not digits  : ['Order', 'A', 'shipped', 'on', '-', '-', 'for', '$', '.']
vowels      : ['e', 'i', 'e', 'o', 'o']


### How many

| Pattern | Means |
|---|---|
| `*` | zero or more |
| `+` | one or more |
| `?` | zero or one, so "optional" |
| `{3}` | exactly three |
| `{2,4}` | between two and four |
| `{2,}` | two or more |

The quantifier applies to whatever comes immediately before it.

In [17]:
codes = ["A1", "A001", "A", "AB12345", "2024-03-01"]

for code in codes:
    is_order_id = bool(re.fullmatch(r"A\d+", code))
    is_date = bool(re.fullmatch(r"\d{4}-\d{2}-\d{2}", code))
    print(f"  {code:<12} A + digits: {is_order_id!s:<6} date shape: {is_date}")

  A1           A + digits: True   date shape: False
  A001         A + digits: True   date shape: False
  A            A + digits: False  date shape: False
  AB12345      A + digits: False  date shape: False
  2024-03-01   A + digits: False  date shape: True


`?` is how you make part of a pattern optional, which is how one pattern copes with two spellings:

In [18]:
for value in ["color", "colour", "colouur"]:
    print(f"  {value:<10} {bool(re.fullmatch(r'colou?r', value))}")

  color      True
  colour     True
  colouur    False


### Where

Anchors match a position rather than a character, so they consume nothing.

| Pattern | Matches at |
|---|---|
| `^` | the start of the string, or of a line with `re.MULTILINE` |
| `$` | the end of the string, or of a line |
| `\b` | a word boundary, the edge between `\w` and not-`\w` |

`\b` is the one that saves you from matching inside a longer word:

In [19]:
text = "cat catalogue concatenate the cat sat"

print("without \\b:", re.findall(r"cat", text))
print("with    \\b:", re.findall(r"\bcat\b", text))

without \b: ['cat', 'cat', 'cat', 'cat']
with    \b: ['cat', 'cat']


### Alternation

`|` means "or", and brackets keep it contained. Without the brackets, the `|` splits the entire pattern:

In [20]:
log = "ERROR disk full / WARNING retry / INFO ok / CRITICAL down"

print("no brackets  :", re.findall(r"^ERROR|WARNING$", log))
print("with brackets:", re.findall(r"\b(ERROR|WARNING|CRITICAL)\b", log))

no brackets  : ['ERROR']
with brackets: ['ERROR', 'WARNING', 'CRITICAL']


---
# 5. Groups: Pulling Out the Pieces

Round brackets do two jobs. They keep part of a pattern together, and they **capture** what matched so you can read it back.

A date is a good first example, because you almost never want the whole thing. You want the year:

In [21]:
found = re.search(r"(\d{4})-(\d{2})-(\d{2})", sample)

if found:
    print("whole match:", found.group())      # group 0
    print("year       :", found.group(1))
    print("month      :", found.group(2))
    print("all groups :", found.groups())

whole match: 2024-03-01
year       : 2024
month      : 03
all groups : ('2024', '03', '01')


Group 0 is always the whole match, and groups are numbered left to right by their opening bracket.

### Named groups

Counting brackets stops being fun at about three. `(?P<name>...)` gives a group a name, and `.groupdict()` hands the whole thing back as a dictionary:

In [22]:
line = "2024-03-01 09:14:22 WARNING Payment retry 1 for order A002"

pattern = r"(?P<date>\S+) (?P<time>\S+) (?P<level>\w+) (?P<message>.*)"
found = re.search(pattern, line)

if found:
    print(found.group("level"))
    for field, value in found.groupdict().items():
        print(f"  {field:<8} {value}")

WARNING
  date     2024-03-01
  time     09:14:22
  level    WARNING
  message  Payment retry 1 for order A002


The pattern is longer, and every reader of it now knows what the fields are. Use named groups for anything you will read again.

### Non-capturing groups

The alternation in section 4 used brackets only to keep the `|` contained. It captured as a side effect, which we never needed. `(?:...)` groups without capturing, which keeps `groups()` and `findall` clean:

In [23]:
text = "orders: A001, A002, B003"

print("capturing    :", re.findall(r"([AB])(\d+)", text))
print("non-capturing:", re.findall(r"(?:[AB])(\d+)", text))

capturing    : [('A', '001'), ('A', '002'), ('B', '003')]
non-capturing: ['001', '002', '003']


### The `findall` trap

`findall` changes what it returns depending on how many groups the pattern has. No groups gives whole matches; one group gives just that group; two or more gives tuples:

In [24]:
print("no groups :", re.findall(r"[AB]\d+", text))
print("one group :", re.findall(r"[AB](\d+)", text))
print("two groups:", re.findall(r"([AB])(\d+)", text))

no groups : ['A001', 'A002', 'B003']
one group : ['001', '002', '003']
two groups: [('A', '001'), ('A', '002'), ('B', '003')]


Nothing warns you. A pattern that returned strings yesterday returns tuples today because someone added a pair of brackets for grouping. If you want the whole match and also need brackets, make them non-capturing, or use `finditer` and read `.group()` yourself.

---
# 6. Greedy and Lazy

Quantifiers are **greedy**: they take as much as they can and only give back what they must. This is the single most surprising thing about regexes, and it shows up the first time anyone tries to match a tag:

In [25]:
html = '<p class="lead">Regex</p> and <b>bold</b>'

print("greedy:", re.findall(r"<.+>", html))

greedy: ['<p class="lead">Regex</p> and <b>bold</b>']


One match, the entire string. `<` matched the first `<`, `.+` ran to the end of the line, then the engine backed up until it found the last `>`. Every intermediate `>` was skipped over on the way.

Adding `?` after a quantifier makes it **lazy**: take as little as possible.

In [26]:
print("lazy  :", re.findall(r"<.+?>", html))

lazy  : ['<p class="lead">', '</p>', '<b>', '</b>']


Four matches, which is what anyone would have expected first time.

The same choice appears in every quantifier: `*?`, `+?`, `??`, `{2,4}?`. When a pattern matches far more than you meant, greed is nearly always the reason.

A third option is often better than either: say what you do not want. `[^>]+` means "anything that is not a closing bracket", which cannot overrun by construction:

In [27]:
print("negated:", re.findall(r"<[^>]+>", html))

negated: ['<p class="lead">', '</p>', '<b>', '</b>']


---
# 7. Compiling, and Flags

`re.compile` turns a pattern into an object you can reuse. The module caches recent patterns anyway, so the reason to compile is readability: the pattern gets a name, and it sits next to its documentation rather than being retyped inside a loop.

In [28]:
HASHTAG = re.compile(r"#\w+")
ORDER_ID = re.compile(r"\b[AB]\d{3}\b")

print(HASHTAG.findall(caption))
print(ORDER_ID.findall("orders A001, B003 and the number 1234"))

['#Python', '#pandas', '#100DaysOfCode']
['A001', 'B003']


Flags change how a pattern is read. Three are worth knowing.

`re.IGNORECASE` does what it says:

In [29]:
print(re.findall(r"python", caption))
print(re.findall(r"python", caption, re.IGNORECASE))

[]
['Python']


`re.MULTILINE` makes `^` and `$` match at the start and end of each line rather than of the whole string:

In [30]:
log_text = """2024-03-01 09:14:22 INFO Order A001 placed
2024-03-01 09:15:03 WARNING Payment retry 1 for order A002
2024-03-01 09:15:40 ERROR Payment failed for order A002
2024-03-01 09:16:00 INFO Order A003 placed"""

print("without MULTILINE:", re.findall(r"^\S+", log_text))
print("with MULTILINE   :", re.findall(r"^\S+", log_text, re.MULTILINE))

without MULTILINE: ['2024-03-01']
with MULTILINE   : ['2024-03-01', '2024-03-01', '2024-03-01', '2024-03-01']


`re.VERBOSE` lets a pattern span several lines with comments, and ignores whitespace inside it. This is how a pattern that anyone has to maintain should be written:

In [31]:
LOG_LINE = re.compile(r"""
    (?P<date>\d{4}-\d{2}-\d{2})     # 2024-03-01
    \s
    (?P<time>\d{2}:\d{2}:\d{2})     # 09:14:22
    \s
    (?P<level>[A-Z]+)               # INFO, WARNING, ERROR
    \s
    (?P<message>.+)                 # the rest of the line
""", re.VERBOSE)

found = LOG_LINE.search(log_text)
if found:
    print(found.groupdict())

{'date': '2024-03-01', 'time': '09:14:22', 'level': 'INFO', 'message': 'Order A001 placed'}


One rule comes with `re.VERBOSE`: ordinary spaces inside the pattern are ignored, so a space you actually want has to be spelled out. The pattern above uses `\s`, which matches any whitespace. For exactly one space, write `[ ]` or `\ `.

### `re.escape`: when the pattern comes from data

Anything you drop into a pattern is read *as* a pattern. A search term typed by a user, or read out of a file, will sooner or later contain a character that means something to the engine:

In [32]:
term = "a.c"                            # a literal string someone typed
haystack = "abc and a.c"

print("used as a pattern:", re.findall(term, haystack))

used as a pattern: ['abc', 'a.c']


The dot matched `b`, so a search for `a.c` reported a hit on `abc`, which does not contain the term at all. `re.escape` puts a backslash in front of every character that carries meaning, so the string matches itself and nothing else:

In [33]:
print("escaped     :", re.escape(term))
print("used escaped:", re.findall(re.escape(term), haystack))

escaped     : a\.c
used escaped: ['a.c']


Use it whenever the pattern is built from a value rather than typed by you. The same applies to a filename, a product code, or anything else with a dot, a bracket or a plus sign in it.

---
# 8. Real-World Examples

### Parsing a log file

The pattern above plus chapter 15's `Counter` turns a log into a summary. This is the shape of nearly every log-processing script.

In [34]:
from collections import Counter, defaultdict

levels = Counter()
by_level = defaultdict(list)

for entry in LOG_LINE.finditer(log_text):
    fields = entry.groupdict()
    levels[fields["level"]] += 1
    by_level[fields["level"]].append(fields["time"])

print("counts:", levels.most_common())
print()
for level, times in sorted(by_level.items()):
    print(f"  {level:<8} {times}")

counts: [('INFO', 2), ('WARNING', 1), ('ERROR', 1)]

  ERROR    ['09:15:40']
  INFO     ['09:14:22', '09:16:00']
  WARNING  ['09:15:03']


### Cleaning a currency column

Amounts arrive from spreadsheets in whatever shape the person typing them felt like. Strip everything that is not a digit or a dot, then convert:

In [35]:
raw_amounts = ["$59.98", "1,299.00 INR", "USD 45", "  89.99  ", "free"]

for raw in raw_amounts:
    digits = re.sub(r"[^\d.]", "", raw)
    print(f"  {raw!r:<16} -> {float(digits) if digits else None}")

  '$59.98'         -> 59.98
  '1,299.00 INR'   -> 1299.0
  'USD 45'         -> 45.0
  '  89.99  '      -> 89.99
  'free'           -> None


### Validating, and knowing when to stop

Regexes are good at rejecting obvious rubbish and bad at deciding what is truly valid. Email is the classic example. Here is the pattern almost everyone writes first:

In [36]:
SIMPLE_EMAIL = re.compile(r"^[\w.]+@\w+\.\w+$")

for address in ["shikhar@example.com", "first.last@example.co.uk",
                "a+tag@example.com", "not-an-email", "@example.com"]:
    print(f"  {address:<28} {bool(SIMPLE_EMAIL.fullmatch(address))}")

  shikhar@example.com          True
  first.last@example.co.uk     False
  a+tag@example.com            False
  not-an-email                 False
  @example.com                 False


It rejects the two obvious failures, and it also rejects `first.last@example.co.uk` and `a+tag@example.com`, both of which are perfectly ordinary addresses that real people use.

The full specification for an email address takes a pattern thousands of characters long, and even that only proves the address is well formed, not that anyone is reading mail there. The practical rule is to check for an `@` with something on each side, and confirm the rest by sending a message.

### Splitting text into sentences and words

In [37]:
review = "The book arrived fast! Was it worth 59.98? Yes... mostly. I'd buy again."

sentences = re.split(r"(?<=[.!?])\s+", review)
for s in sentences:
    print(f"  {s}")

print("\nwords:", re.findall(r"\b[\w']+\b", review.lower())[:8], "...")

  The book arrived fast!
  Was it worth 59.98?
  Yes...
  mostly.
  I'd buy again.

words: ['the', 'book', 'arrived', 'fast', 'was', 'it', 'worth', '59'] ...


`(?<=[.!?])` is a **lookbehind**: it requires a `.`, `!` or `?` just before the split point without consuming it, so the punctuation stays attached to the sentence it ends. Notice that `Yes...` split correctly and `59.98` did not, because the dot inside the number is not followed by whitespace.

The word pattern is a separate question, and it does break `59.98` into `59` and `98`, since a dot is not a word character. Whether that is right depends on what you are counting. A pattern makes that decision for you quietly, so it is worth looking at the output rather than assuming.

---
# 9. When a Regex Is the Wrong Tool

A regex can match anything that is regular in shape. Plenty of everyday text is not.

### Do not parse HTML with a regex

The lazy pattern from section 6 looked convincing. Give it real HTML and it falls apart:

In [38]:
page = '<a href="/p1">first</a> <!-- <a href="/hidden">skip</a> --> <a\n   href="/p2">second</a>'

print("regex   :", re.findall(r'<a href="([^"]+)">', page))

regex   : ['/p1', '/hidden']


Three `<a>` tags, two matches, and two mistakes, neither of which announces itself. It returned `/hidden`, which sits inside an HTML comment and is not a link on the page at all. And it missed `/p2` entirely, because that tag has a newline between `<a` and `href`. Fixing those two cases means adding rules for comments, attribute order, quoting styles and whitespace, and every fix opens the next gap.

An HTML parser already knows all of that. Chapter 18 uses BeautifulSoup, which handles this page correctly without a pattern in sight.

### Do not parse CSV with a regex

Chapter 10's `csv` module exists for one reason, and this is it:

In [39]:
row = 'A001,"Sharma, Alice",Books,59.98'

print("str.split:", row.split(","))
print("re.split :", re.split(r",", row))

import csv
import io

print("csv      :", next(csv.reader(io.StringIO(row))))

str.split: ['A001', '"Sharma', ' Alice"', 'Books', '59.98']
re.split : ['A001', '"Sharma', ' Alice"', 'Books', '59.98']
csv      : ['A001', 'Sharma, Alice', 'Books', '59.98']


The comma inside the quoted name is part of the name. Only the `csv` module knows that.

### Do not use a regex where a string method will do

`re.search(r"^https://", url)` is a slower and less readable way of writing `url.startswith("https://")`. String methods are clearer for fixed text:

| Instead of | Write |
|---|---|
| `re.search(r"^abc", s)` | `s.startswith("abc")` |
| `re.search(r"abc$", s)` | `s.endswith("abc")` |
| `re.search(r"abc", s)` | `"abc" in s` |
| `re.sub(r"abc", "x", s)` | `s.replace("abc", "x")` |

Reach for `re` when the thing you are describing is a *shape*, not a fixed string.

### One warning we will not run

Some patterns take exponential time on input that nearly matches. `(a+)+b` against a long run of `a` characters is the standard example: it can hang for minutes on a string of forty characters. This is called catastrophic backtracking, and it is a real way to take a web service down.

The cell would never finish, so this chapter does not run it. Avoiding it is mostly a matter of not nesting quantifiers, so prefer `[^>]+` over `.+?` and never put a `+` directly around a group that already has one.

---
# 10. Redoing Chapter 2, Properly

Everything the chapter opened with, in the version you would actually ship.

In [40]:
HASHTAG = re.compile(r"#\w+")
MENTION = re.compile(r"@\w+")

def parse_caption(text):
    return {"hashtags": HASHTAG.findall(text),
            "mentions": MENTION.findall(text)}


def slugify(text):
    return re.sub(r"[^a-z0-9]+", "-", text.lower()).strip("-")


print(parse_caption(caption))
print(parse_caption("No tags here, just words."))
print()

for q in ["How to learn Python in 2026?  (beginner guide)",
          "  Data Science & AI: where to start!  ",
          "already-a-slug"]:
    print(f"  {q!r:<48} -> {slugify(q)!r}")

{'hashtags': ['#Python', '#pandas', '#100DaysOfCode'], 'mentions': ['@guido', '@raymondh']}
{'hashtags': [], 'mentions': []}

  'How to learn Python in 2026?  (beginner guide)' -> 'how-to-learn-python-in-2026-beginner-guide'
  '  Data Science & AI: where to start!  '         -> 'data-science-ai-where-to-start'
  'already-a-slug'                                 -> 'already-a-slug'


`parse_caption` returns empty lists rather than failing when there is nothing to find, and `slugify` copes with punctuation, repeated spaces, leading and trailing space, and text that is already a slug. Neither needed a special case for any of them.

---
# 11. Common Mistakes

| Mistake | What happens | Fix |
|---|---|---|
| Pattern written without `r"..."` | Silently matches nothing | Always use a raw string |
| `.group()` on a `search` result | `AttributeError` on `None` | Check the match first |
| Using `re.match` to search | Only ever looks at the start | Use `re.search` |
| Adding brackets for grouping | `findall` starts returning tuples | Use `(?:...)` |
| `.+` where the text has delimiters | Matches far too much | Use `.+?` or `[^x]+` |
| `re.split` on CSV | Breaks on quoted commas | Use the `csv` module |
| A regex for HTML | Misses comments, newlines, attribute order | Use a parser, as in chapter 18 |
| A pattern built from a value | Punctuation in the value acts as a pattern | Wrap it in `re.escape` |
| Nested quantifiers such as `(a+)+` | Can hang on near-matches | Do not nest quantifiers |

---
# 12. Summary: Your Regex Cheat Sheet

**The functions**

```python
import re

re.search(pattern, text)      # first match object, or None
re.match(pattern, text)       # same, but anchored at the start
re.fullmatch(pattern, text)   # the whole string must match
re.findall(pattern, text)     # list of strings
re.finditer(pattern, text)    # iterator of match objects
re.sub(pattern, repl, text)   # replace; repl may be a function
re.split(pattern, text)       # split on a pattern
re.compile(pattern, flags)    # a reusable pattern object
re.escape(text)               # treat text as literal, not as a pattern
```

**The match object**

```python
found.group()        # the whole match
found.group(1)       # the first capture group
found.group("name")  # a named group
found.groups()       # every group as a tuple
found.groupdict()    # named groups as a dict
found.start(), found.end(), found.span()
```

**What to match**

| | | | |
|---|---|---|---|
| `\d` digit | `\w` word char | `\s` whitespace | `.` any char |
| `\D` not digit | `\W` not word | `\S` not space | `[abc]` one of |
| `[a-z]` a range | `[^abc]` not these | `\|` or | `(?:...)` group only |

**How many**

| | | | | | |
|---|---|---|---|---|---|
| `*` 0+ | `+` 1+ | `?` 0 or 1 | `{3}` exactly | `{2,4}` between | `+?` lazy |

**Where**

| | | |
|---|---|---|
| `^` start | `$` end | `\b` word boundary |

**Groups**

```python
r"(\d{4})-(\d{2})"                    # numbered: .group(1), .group(2)
r"(?P<year>\d{4})-(?P<month>\d{2})"   # named: .group("year"), .groupdict()
r"(?:ERROR|WARNING)"                  # grouped, not captured
```

**Flags**

| Flag | Effect |
|---|---|
| `re.IGNORECASE` | case does not matter |
| `re.MULTILINE` | `^` and `$` match on every line |
| `re.VERBOSE` | whitespace ignored, `#` starts a comment |

**Patterns you will write again**

```python
r"#\w+"                             # hashtag
r"@\w+"                             # mention
r"\d{4}-\d{2}-\d{2}"                # ISO date
r"[^a-z0-9]+"                       # runs of punctuation, for slugs
r"[^\d.]"                           # everything that is not part of a number
r"<[^>]+>"                          # a tag, without overrunning
```

---

**Next:** chapter 17 goes to the network. We fetch real data over HTTP with `requests`, which is the first package in this course that needs `pip install`, and chapter 11's virtual environment stops being theory.